# Identifying candidate Pleiades members with DBSCAN

This notebook explores whether Gaia sources form a compact overdensity in the three-dimensional astrometric space defined by parallax, proper motion in right ascension (`pmra`), and proper motion in declination (`pmdec`).

DBSCAN is a density-based clustering algorithm. It can identify dense groups without specifying the number of clusters in advance and can label isolated points as noise. However, its result depends on feature scaling and on two parameters, `eps` and `min_samples`. We will examine those choices rather than treating its first output as a final membership catalogue.

## 1. Imports and project paths

We use NumPy to construct the numerical feature matrix, Astropy to preserve the Gaia table and its units, and scikit-learn's `StandardScaler` to place the three features on comparable numerical scales.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

print(f"Python interpreter: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

In [ ]:
project_root = Path.cwd()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

cleaned_data_path = (
    project_root
    / "data"
    / "processed"
    / "gaia_dr3_pleiades_3deg_cleaned.fits"
)
processed_data_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"

processed_data_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Processed catalogue exists: {cleaned_data_path.exists()}")

## 2. Load the cleaned catalogue

This is the derived catalogue produced by Notebook 02. It contains finite values for the three clustering features and satisfies the documented astrometric quality criteria. Loading it into a new variable does not alter the saved FITS file.

In [ ]:
if not cleaned_data_path.exists():
    raise FileNotFoundError(
        "The cleaned catalogue is missing. Run Notebook 02 first."
    )

gaia_clean = Table.read(cleaned_data_path, format="fits")

print(f"Rows: {len(gaia_clean):,}")
print(f"Columns: {len(gaia_clean.colnames)}")

## 3. Construct the astrometric feature matrix

Scikit-learn expects a two-dimensional NumPy array with one source per row and one feature per column. We keep the feature names separately so that the column order remains explicit.

The row order of `features` is exactly the row order of `gaia_clean`. This correspondence will later allow us to attach each DBSCAN label to the correct Gaia `source_id`.

In [ ]:
feature_names = ["parallax", "pmra", "pmdec"]

features = np.column_stack(
    [np.asarray(gaia_clean[name], dtype=float) for name in feature_names]
)

assert features.shape == (len(gaia_clean), len(feature_names))
assert np.all(np.isfinite(features))

print(f"Feature matrix shape: {features.shape}")
print(f"Feature order: {feature_names}")

In [ ]:
unscaled_summary = Table(
    rows=[
        (
            name,
            np.mean(features[:, index]),
            np.std(features[:, index]),
            np.min(features[:, index]),
            np.max(features[:, index]),
        )
        for index, name in enumerate(feature_names)
    ],
    names=("feature", "mean", "standard_deviation", "minimum", "maximum"),
)

for name in unscaled_summary.colnames[1:]:
    unscaled_summary[name].format = ".3f"

unscaled_summary

## 4. Standardize the features

DBSCAN measures distances between points. In the original units, proper-motion values span a much wider numerical range than parallax, so proper motion would dominate the distance calculation.

`StandardScaler` transforms each feature by subtracting its sample mean and dividing by its sample standard deviation:

$$z = \frac{x - \bar{x}}{s}. $$

After this transformation, every feature has mean approximately 0 and standard deviation approximately 1. This makes a one-standard-deviation change comparable across the three axes. It does not imply that the measurement uncertainties are equal, and it does not remove physical outliers.

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

scaled_summary = Table(
    rows=[
        (
            name,
            np.mean(scaled_features[:, index]),
            np.std(scaled_features[:, index]),
        )
        for index, name in enumerate(feature_names)
    ],
    names=("feature", "scaled_mean", "scaled_standard_deviation"),
)

for name in scaled_summary.colnames[1:]:
    scaled_summary[name].format = ".6f"

scaled_summary

### Scaling checkpoint

The scaled means should be extremely close to 0 and the standard deviations should be 1. Small residual values near zero arise from floating-point arithmetic.

Standardization is a modelling choice rather than a physical correction. Before fitting DBSCAN, we will next examine neighbour distances to choose a defensible starting range for `eps`.

## 5. Examine nearest-neighbour distances

DBSCAN uses two linked parameters:

- `min_samples` is the minimum number of points, including the source itself, required inside a neighbourhood for that source to be considered a dense core point.
- `eps` is the radius of that neighbourhood in scaled feature space.

We begin with `min_samples = 10` as an exploratory value. It is larger than the minimum needed to define density in three dimensions, so a cluster must have more support than a very small accidental grouping. It is not yet a final choice.

For every source, we calculate the distance to its 10th nearest point. Because the source itself is returned at distance zero, this corresponds to the source plus nine other nearby sources—the same counting convention used by DBSCAN.

In [ ]:
min_samples_start = 10

nearest_neighbours = NearestNeighbors(
    n_neighbors=min_samples_start,
    n_jobs=-1,
)
nearest_neighbours.fit(scaled_features)
neighbour_distances, neighbour_indices = nearest_neighbours.kneighbors(
    scaled_features
)

kth_neighbour_distances = neighbour_distances[:, -1]
sorted_kth_distances = np.sort(kth_neighbour_distances)

distance_percentiles = Table(
    rows=[
        (percentile, np.percentile(kth_neighbour_distances, percentile))
        for percentile in [25, 50, 75, 90, 95, 99, 99.5]
    ],
    names=("percentile", "10th_neighbour_distance"),
)
distance_percentiles["10th_neighbour_distance"].format = ".4f"
distance_percentiles

### The sorted-distance diagnostic

Sorting these distances from smallest to largest produces a *k-distance plot*. Dense regions appear on the relatively flat left-hand portion because their sources have nearby neighbours. Sparse field regions and isolated sources rise on the right. A bend or elbow can suggest a starting `eps`: values below it tend to describe dense structure, while larger values increasingly connect sparse regions.

The first panel uses a logarithmic y-axis so that the full range remains visible. The second panel magnifies the densest 99.5% of sources. The broad Galactic field does not have a single uniform density, so we should not expect a perfectly sharp elbow.

In [ ]:
source_rank = np.arange(1, len(sorted_kth_distances) + 1)
zoom_count = int(0.995 * len(sorted_kth_distances))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(source_rank, sorted_kth_distances, color="tab:blue")
axes[0].set_yscale("log")
axes[0].set(
    xlabel="Sources ordered by neighbour distance",
    ylabel="10th-neighbour distance (scaled units)",
    title="Full distance range",
)

axes[1].plot(
    source_rank[:zoom_count],
    sorted_kth_distances[:zoom_count],
    color="tab:orange",
)
axes[1].set(
    xlabel="Sources ordered by neighbour distance",
    ylabel="10th-neighbour distance (scaled units)",
    title="Densest 99.5% of sources",
)

distance_figure_path = figures_dir / "03_tenth_neighbour_distances.png"
fig.savefig(distance_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {distance_figure_path}")

### Interpretation checkpoint

Look for where the magnified curve begins rising more rapidly rather than searching for one exact magical value. We will use several nearby `eps` values in the next section and compare their scientific consequences: the number of clusters, the number of noise points, and whether a compact group appears near the expected Pleiades astrometry.

## 6. Define a broad local analysis window

The full catalogue is scientifically useful because it shows the Pleiades against the Galactic field. However, the neighbour-distance curve also shows that this field has strongly varying density. A single DBSCAN `eps` is poorly suited to describing both the extremely dense central field and a much smaller cluster far from it. Running DBSCAN on all 214,953 sources would also create unnecessarily large neighbourhood calculations.

We therefore define a **broad exploratory analysis window** around the compact feature seen in Notebook 02:

- parallax from 2 to 12 mas,
- `pmra` from 0 to 40 mas/yr,
- `pmdec` from -70 to -25 mas/yr.

These limits are intentionally much wider than the visible compact overdensity. They preserve a surrounding comparison population for DBSCAN rather than declaring every source inside the box a member. Nevertheless, this is an exploratory, Pleiades-informed choice: DBSCAN cannot recover a member excluded by the window, so we must later test sensitivity to these boundaries.

In [ ]:
analysis_limits = {
    "parallax": (2.0, 12.0),
    "pmra": (0.0, 40.0),
    "pmdec": (-70.0, -25.0),
}

analysis_mask = np.ones(len(gaia_clean), dtype=bool)

for name, (lower, upper) in analysis_limits.items():
    values = np.asarray(gaia_clean[name], dtype=float)
    analysis_mask &= (values >= lower) & (values <= upper)

gaia_analysis = gaia_clean[analysis_mask].copy()

print(f"Sources in the cleaned catalogue: {len(gaia_clean):,}")
print(f"Sources in the broad analysis window: {len(gaia_analysis):,}")
print(f"Fraction retained for local DBSCAN: {len(gaia_analysis) / len(gaia_clean):.2%}")

### Scale the local analysis sample

The mean and standard deviation of the broad field are not representative of this smaller region, so we fit a new scaler using only the local analysis sample. This ensures that the three coordinates contribute comparably within the actual dataset supplied to DBSCAN.

We give the new variables an `analysis_` prefix to avoid accidentally mixing the full-catalogue and local arrays.

In [ ]:
analysis_features = np.column_stack(
    [
        np.asarray(gaia_analysis[name], dtype=float)
        for name in feature_names
    ]
)

analysis_scaler = StandardScaler()
analysis_scaled_features = analysis_scaler.fit_transform(analysis_features)

assert analysis_features.shape == (len(gaia_analysis), 3)
assert np.all(np.isfinite(analysis_scaled_features))

print(f"Local feature matrix shape: {analysis_features.shape}")
print("Scaled means:", np.mean(analysis_scaled_features, axis=0))
print("Scaled standard deviations:", np.std(analysis_scaled_features, axis=0))

### Analysis-window checkpoint

Being inside this window does **not** imply membership. It only defines the local population in which DBSCAN will search for a denser structure. In the next section, we will recalculate neighbour distances for these local scaled features and then compare several nearby DBSCAN parameter choices.

## 7. Recalculate neighbour distances locally

The earlier k-distance curve described the entire Galactic field. Because DBSCAN will now operate on `analysis_scaled_features`, its useful `eps` scale must be estimated from those same data. We repeat the 10th-neighbour calculation for the local sample.

In [ ]:
local_neighbours = NearestNeighbors(
    n_neighbors=min_samples_start,
    n_jobs=-1,
)
local_neighbours.fit(analysis_scaled_features)
local_distances, local_indices = local_neighbours.kneighbors(
    analysis_scaled_features
)

local_kth_distances = local_distances[:, -1]
local_sorted_distances = np.sort(local_kth_distances)

local_distance_summary = Table(
    rows=[
        (percentile, np.percentile(local_kth_distances, percentile))
        for percentile in [10, 25, 50, 75, 90, 95, 99]
    ],
    names=("percentile", "10th_neighbour_distance"),
)
local_distance_summary["10th_neighbour_distance"].format = ".4f"
local_distance_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
ax.plot(
    np.arange(1, len(local_sorted_distances) + 1),
    local_sorted_distances,
    color="tab:blue",
)
ax.set(
    xlabel="Local sources ordered by neighbour distance",
    ylabel="10th-neighbour distance (scaled units)",
    title="Local 10th-neighbour distances",
)

local_distance_figure_path = figures_dir / "03_local_tenth_neighbour_distances.png"
fig.savefig(local_distance_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {local_distance_figure_path}")

## 8. Compare initial DBSCAN parameters

The local curve still bends gradually, so one `eps` value cannot be justified uniquely from the plot. We therefore hold `min_samples = 10` fixed and compare several radii from 0.08 to 0.35 scaled units.

For each run we record:

- the number of clusters, excluding the noise label `-1`,
- the number and percentage of sources labelled as noise,
- the size of the largest cluster.

A larger `eps` connects more sources, so noise generally decreases and separate groups may merge. The largest cluster is not automatically the Pleiades; this table only describes statistical behaviour.

In [ ]:
eps_values = [0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30, 0.35]
dbscan_labels_by_eps = {}
parameter_rows = []

for eps in eps_values:
    model = DBSCAN(
        eps=eps,
        min_samples=min_samples_start,
        n_jobs=-1,
    )
    labels = model.fit_predict(analysis_scaled_features)
    dbscan_labels_by_eps[eps] = labels

    cluster_labels = np.unique(labels[labels != -1])
    cluster_sizes = [
        np.count_nonzero(labels == label) for label in cluster_labels
    ]
    noise_count = np.count_nonzero(labels == -1)
    largest_cluster_size = max(cluster_sizes, default=0)

    parameter_rows.append(
        (
            eps,
            len(cluster_labels),
            noise_count,
            100 * noise_count / len(labels),
            largest_cluster_size,
        )
    )

parameter_summary = Table(
    rows=parameter_rows,
    names=("eps", "n_clusters", "noise_sources", "noise_percent", "largest_cluster"),
)
parameter_summary["eps"].format = ".2f"
parameter_summary["noise_percent"].format = ".1f"
parameter_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4), constrained_layout=True)

axes[0].plot(
    parameter_summary["eps"],
    parameter_summary["noise_percent"],
    marker="o",
)
axes[0].set(
    xlabel="eps (scaled units)",
    ylabel="Noise sources (%)",
    title="Noise fraction",
)

axes[1].plot(
    parameter_summary["eps"],
    parameter_summary["largest_cluster"],
    marker="o",
    color="tab:orange",
)
axes[1].set(
    xlabel="eps (scaled units)",
    ylabel="Sources in largest cluster",
    title="Largest-cluster size",
)

parameter_figure_path = figures_dir / "03_initial_dbscan_parameter_comparison.png"
fig.savefig(parameter_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {parameter_figure_path}")

### Parameter-comparison checkpoint

We should look for a range where the candidate cluster remains fairly stable, rather than choosing the run with the most members. A sudden jump in the largest-cluster size can indicate that `eps` has begun joining the cluster to the surrounding field. Next, we will locate the cluster near the expected Pleiades astrometry in every run and compare its centre and size directly.

## 9. Locate the Pleiades-like cluster

DBSCAN labels such as 0 or 1 have no physical meaning; they are assigned in the order the algorithm encounters groups. We identify the astrophysically relevant cluster by comparing each cluster's median astrometry with an approximate reference location derived from the compact feature visible in Notebook 02:

$$\left(\varpi, \mu_{\alpha*}, \mu_{\delta}\right) \approx (7.3, 20, -45).$$

The units are mas for parallax and mas/yr for proper motion. This reference is used only to decide which DBSCAN label to follow across runs; it does not impose a membership box around individual sources.

For each `eps`, we calculate every cluster's median coordinates and select the median closest to the reference in the locally scaled feature space. Using scaled distance prevents the feature with the largest numerical units from dominating this comparison.

In [ ]:
pleiades_reference = np.array([7.3, 20.0, -45.0])
reference_scaled = analysis_scaler.transform(
    pleiades_reference.reshape(1, -1)
)[0]

candidate_masks_by_eps = {}
candidate_rows = []
previous_candidate_mask = None

for eps in eps_values:
    labels = dbscan_labels_by_eps[eps]
    cluster_labels = np.unique(labels[labels != -1])
    cluster_comparisons = []

    for label in cluster_labels:
        cluster_mask = labels == label
        cluster_median = np.median(
            analysis_features[cluster_mask], axis=0
        )
        median_scaled = analysis_scaler.transform(
            cluster_median.reshape(1, -1)
        )[0]
        distance_to_reference = np.linalg.norm(
            median_scaled - reference_scaled
        )
        cluster_comparisons.append(
            (distance_to_reference, label, cluster_mask, cluster_median)
        )

    _, candidate_label, candidate_mask, candidate_median = min(
        cluster_comparisons, key=lambda item: item[0]
    )
    candidate_masks_by_eps[eps] = candidate_mask

    if previous_candidate_mask is None:
        overlap_with_previous = np.nan
    else:
        intersection = np.count_nonzero(
            candidate_mask & previous_candidate_mask
        )
        union = np.count_nonzero(
            candidate_mask | previous_candidate_mask
        )
        overlap_with_previous = intersection / union

    candidate_rows.append(
        (
            eps,
            candidate_label,
            np.count_nonzero(candidate_mask),
            *candidate_median,
            overlap_with_previous,
        )
    )
    previous_candidate_mask = candidate_mask

candidate_summary = Table(
    rows=candidate_rows,
    names=(
        "eps",
        "label",
        "candidate_sources",
        "median_parallax",
        "median_pmra",
        "median_pmdec",
        "overlap_with_previous",
    ),
)

for name in candidate_summary.colnames[3:]:
    candidate_summary[name].format = ".3f"
candidate_summary["eps"].format = ".2f"
candidate_summary

### Membership overlap

`overlap_with_previous` is the Jaccard similarity between consecutive candidate sets:

$$J(A,B)=\frac{|A\cap B|}{|A\cup B|}. $$

A value near 1 means almost the same sources were selected at two neighbouring `eps` values. This is more informative than comparing sample sizes alone: two runs could contain the same number of sources but select different objects.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)

axes[0, 0].plot(
    candidate_summary["eps"],
    candidate_summary["candidate_sources"],
    marker="o",
)
axes[0, 0].set(ylabel="Candidate sources", title="Candidate size")

median_columns = ["median_parallax", "median_pmra", "median_pmdec"]
reference_values = pleiades_reference
axis_labels = ["Median parallax (mas)", "Median pmra (mas/yr)", "Median pmdec (mas/yr)"]

for ax, column, reference, ylabel in zip(
    axes.flat[1:], median_columns, reference_values, axis_labels
):
    ax.plot(candidate_summary["eps"], candidate_summary[column], marker="o")
    ax.axhline(reference, color="tab:red", linestyle="--", alpha=0.7)
    ax.set_ylabel(ylabel)

for ax in axes.flat:
    ax.set_xlabel("eps (scaled units)")

stability_figure_path = figures_dir / "03_candidate_cluster_stability.png"
fig.savefig(stability_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {stability_figure_path}")

## 10. Choose a provisional baseline `eps`

Across the middle of the tested range, the candidate median remains near roughly 7.36 mas, 19.85 mas/yr, and -45.38 mas/yr, while consecutive membership overlaps are very high. We adopt `eps = 0.18` as a provisional baseline because it lies inside this stable region rather than at either edge.

This is not yet a final membership decision. We still need to test `min_samples`, inspect the candidate distribution directly, consider possible field contamination, and validate the sequence with a colour–magnitude diagram.

In [ ]:
baseline_eps = 0.18
baseline_labels = dbscan_labels_by_eps[baseline_eps]
baseline_candidate_mask = candidate_masks_by_eps[baseline_eps]

print(f"Provisional baseline eps: {baseline_eps:.2f}")
print(f"Candidate sources: {np.count_nonzero(baseline_candidate_mask):,}")

## 11. Test sensitivity to `min_samples`

With `eps = 0.18` fixed, we now vary `min_samples`. Lower values allow small local groups to become clusters more easily; higher values require stronger density support and tend to classify more peripheral sources as noise.

For each run, we again locate the cluster nearest the Pleiades reference. We compare it with the provisional baseline candidate using Jaccard similarity. This tests whether the candidate is a persistent dense structure or an artefact of choosing exactly `min_samples = 10`.

In [ ]:
min_samples_values = [5, 8, 10, 12, 15, 20, 30, 50]
min_samples_labels = {}
min_samples_candidate_masks = {}
min_samples_rows = []

for min_samples in min_samples_values:
    labels = DBSCAN(
        eps=baseline_eps,
        min_samples=min_samples,
        n_jobs=-1,
    ).fit_predict(analysis_scaled_features)
    min_samples_labels[min_samples] = labels

    cluster_labels = np.unique(labels[labels != -1])
    cluster_comparisons = []

    for label in cluster_labels:
        cluster_mask = labels == label
        cluster_median = np.median(
            analysis_features[cluster_mask], axis=0
        )
        median_scaled = analysis_scaler.transform(
            cluster_median.reshape(1, -1)
        )[0]
        distance_to_reference = np.linalg.norm(
            median_scaled - reference_scaled
        )
        cluster_comparisons.append(
            (distance_to_reference, cluster_mask, cluster_median)
        )

    _, candidate_mask, candidate_median = min(
        cluster_comparisons, key=lambda item: item[0]
    )
    min_samples_candidate_masks[min_samples] = candidate_mask

    intersection = np.count_nonzero(
        candidate_mask & baseline_candidate_mask
    )
    union = np.count_nonzero(
        candidate_mask | baseline_candidate_mask
    )
    overlap_with_baseline = intersection / union
    noise_count = np.count_nonzero(labels == -1)

    min_samples_rows.append(
        (
            min_samples,
            len(cluster_labels),
            np.count_nonzero(candidate_mask),
            100 * noise_count / len(labels),
            *candidate_median,
            overlap_with_baseline,
        )
    )

min_samples_summary = Table(
    rows=min_samples_rows,
    names=(
        "min_samples",
        "n_clusters",
        "candidate_sources",
        "noise_percent",
        "median_parallax",
        "median_pmra",
        "median_pmdec",
        "overlap_with_baseline",
    ),
)

for name in min_samples_summary.colnames[3:]:
    min_samples_summary[name].format = ".3f"
min_samples_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4), constrained_layout=True)

axes[0].plot(
    min_samples_summary["min_samples"],
    min_samples_summary["candidate_sources"],
    marker="o",
)
axes[0].axvline(10, color="tab:red", linestyle="--", label="Baseline")
axes[0].set(
    xlabel="min_samples",
    ylabel="Candidate sources",
    title="Candidate size",
)
axes[0].legend()

axes[1].plot(
    min_samples_summary["min_samples"],
    min_samples_summary["overlap_with_baseline"],
    marker="o",
    color="tab:green",
)
axes[1].set(
    xlabel="min_samples",
    ylabel="Jaccard overlap with baseline",
    ylim=(0.9, 1.01),
    title="Membership stability",
)

min_samples_figure_path = figures_dir / "03_min_samples_sensitivity.png"
fig.savefig(min_samples_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {min_samples_figure_path}")

### `min_samples` conclusion

The candidate remains especially stable around `min_samples = 8` to 15: its median astrometry changes very little and its membership overlap with the baseline remains close to 1. Even substantially larger values mainly trim the candidate boundary rather than moving its centre.

We therefore retain `min_samples = 10` as a reasonable provisional value. The next step is visual inspection of the baseline candidates and non-candidates in parallax and proper-motion space.

## 12. Inspect the baseline candidate

We now separate the local analysis table into the provisional DBSCAN candidate and the remaining local comparison sources. This is still an exploratory classification: `gaia_candidate` is not yet a validated membership catalogue.

We summarize each candidate feature with its median and 16th and 84th percentiles. For a roughly bell-shaped distribution, this interval is analogous to about one standard deviation around the centre, but here it is simply a robust description of the observed spread. It combines intrinsic cluster structure, measurement uncertainty, perspective effects, binaries, and possible contaminants.

In [ ]:
gaia_candidate = gaia_analysis[baseline_candidate_mask].copy()
gaia_local_comparison = gaia_analysis[~baseline_candidate_mask].copy()

candidate_summary_rows = []

for name in feature_names:
    values = np.asarray(gaia_candidate[name], dtype=float)
    percentile_16, median, percentile_84 = np.percentile(
        values, [16, 50, 84]
    )
    candidate_summary_rows.append(
        (name, percentile_16, median, percentile_84)
    )

baseline_candidate_summary = Table(
    rows=candidate_summary_rows,
    names=("feature", "percentile_16", "median", "percentile_84"),
)

for name in baseline_candidate_summary.colnames[1:]:
    baseline_candidate_summary[name].format = ".3f"

print(f"Baseline candidates: {len(gaia_candidate):,}")
print(f"Other sources in the local window: {len(gaia_local_comparison):,}")
baseline_candidate_summary

### Astrometric and spatial diagnostic plots

The first two panels show projections of the three-dimensional feature space used by DBSCAN. The third shows sky coordinates, which were **not** supplied to the algorithm. Spatial concentration in that panel is therefore an independent qualitative check rather than a built-in consequence of clustering.

The grey points are other sources inside the broad analysis window, not the entire Gaia query.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

axes[0].scatter(
    gaia_local_comparison["pmra"],
    gaia_local_comparison["pmdec"],
    s=8,
    color="lightgrey",
    alpha=0.6,
    label="Other local sources",
)
axes[0].scatter(
    gaia_candidate["pmra"],
    gaia_candidate["pmdec"],
    s=9,
    color="tab:blue",
    alpha=0.75,
    label="DBSCAN candidates",
)
axes[0].set(
    xlabel=r"$\mu_{\alpha*}$ (mas yr$^{-1}$)",
    ylabel=r"$\mu_{\delta}$ (mas yr$^{-1}$)",
    title="Proper-motion plane",
)
axes[0].legend(markerscale=2)

axes[1].scatter(
    gaia_local_comparison["parallax"],
    gaia_local_comparison["pmdec"],
    s=8,
    color="lightgrey",
    alpha=0.6,
)
axes[1].scatter(
    gaia_candidate["parallax"],
    gaia_candidate["pmdec"],
    s=9,
    color="tab:blue",
    alpha=0.75,
)
axes[1].set(
    xlabel="Parallax (mas)",
    ylabel=r"$\mu_{\delta}$ (mas yr$^{-1}$)",
    title="Parallax-proper-motion projection",
)

axes[2].scatter(
    gaia_local_comparison["ra"],
    gaia_local_comparison["dec"],
    s=8,
    color="lightgrey",
    alpha=0.5,
)
axes[2].scatter(
    gaia_candidate["ra"],
    gaia_candidate["dec"],
    s=9,
    color="tab:blue",
    alpha=0.75,
)
axes[2].set(
    xlabel="Right ascension (deg)",
    ylabel="Declination (deg)",
    title="Sky distribution (not clustered)",
)
axes[2].invert_xaxis()

candidate_figure_path = figures_dir / "03_baseline_candidate_diagnostics.png"
fig.savefig(candidate_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {candidate_figure_path}")

### Questions for interpretation

1. Do the candidates form a compact and continuous structure in both astrometric projections?
2. Are there blue points stretching toward the grey field population that might represent border points or contaminants?
3. Does the sky distribution show a central concentration near the approximate Pleiades centre at RA = 56.75 deg and Dec = 24.12 deg?

A convincing answer to the third question supports the identification, because RA and Dec did not influence DBSCAN. It still does not prove every blue source is a member.

## 13. Distinguish core and border candidates

DBSCAN assigns clustered sources one of two internal roles:

- A **core point** has at least `min_samples` sources, including itself, inside radius `eps`. It directly satisfies the adopted density requirement.
- A **border point** lies within `eps` of a core point but does not independently have enough neighbours to be core. It belongs to the connected cluster but has weaker local density support.

Both types receive the same cluster label. Recording the distinction lets later analysis treat peripheral candidates with appropriate caution; it is not a probability of membership.

In [ ]:
baseline_model = DBSCAN(
    eps=baseline_eps,
    min_samples=min_samples_start,
    n_jobs=-1,
).fit(analysis_scaled_features)

baseline_labels = baseline_model.labels_
candidate_labels = np.unique(
    baseline_labels[baseline_candidate_mask]
)
assert len(candidate_labels) == 1
baseline_candidate_label = int(candidate_labels[0])

is_core_source = np.zeros(len(gaia_analysis), dtype=bool)
is_core_source[baseline_model.core_sample_indices_] = True
candidate_is_core = is_core_source[baseline_candidate_mask]

print(f"Candidate DBSCAN label: {baseline_candidate_label}")
print(f"Core candidates: {np.count_nonzero(candidate_is_core):,}")
print(f"Border candidates: {np.count_nonzero(~candidate_is_core):,}")

## 14. Save the provisional candidate catalogue

We create a new table containing the original Gaia columns plus two DBSCAN result columns:

- `dbscan_label` records the numerical cluster label from this run.
- `dbscan_core` is `True` for core points and `False` for border points.

The adopted parameters and analysis-window limits are also stored in the FITS header as metadata. This file remains a **provisional candidate catalogue** until the colour–magnitude and published-catalogue checks are complete.

In [ ]:
candidate_catalogue = gaia_analysis[baseline_candidate_mask].copy()
candidate_catalogue["dbscan_label"] = baseline_labels[
    baseline_candidate_mask
]
candidate_catalogue["dbscan_core"] = candidate_is_core

candidate_catalogue.meta["DBEPS"] = baseline_eps
candidate_catalogue.meta["DBMIN"] = min_samples_start
candidate_catalogue.meta["PLXMIN"] = analysis_limits["parallax"][0]
candidate_catalogue.meta["PLXMAX"] = analysis_limits["parallax"][1]
candidate_catalogue.meta["PMRMIN"] = analysis_limits["pmra"][0]
candidate_catalogue.meta["PMRMAX"] = analysis_limits["pmra"][1]
candidate_catalogue.meta["PMDMIN"] = analysis_limits["pmdec"][0]
candidate_catalogue.meta["PMDMAX"] = analysis_limits["pmdec"][1]

candidate_data_path = (
    processed_data_dir
    / "gaia_dr3_pleiades_dbscan_candidates.fits"
)
candidate_catalogue.write(
    candidate_data_path, format="fits", overwrite=True
)

print(f"Saved {len(candidate_catalogue):,} provisional candidates to:")
print(candidate_data_path)

### Validate the saved file

We read the file back from disk and verify its row count, unique source identifiers, result columns, parameter metadata, and core/border totals. This checks the actual artifact that Notebook 04 will use, not only the in-memory table.

In [ ]:
saved_candidates = Table.read(candidate_data_path, format="fits")

assert len(saved_candidates) == len(candidate_catalogue)
assert len(np.unique(saved_candidates["source_id"])) == len(saved_candidates)
assert "dbscan_label" in saved_candidates.colnames
assert "dbscan_core" in saved_candidates.colnames
assert np.all(saved_candidates["dbscan_label"] == baseline_candidate_label)
assert saved_candidates.meta["DBEPS"] == baseline_eps
assert saved_candidates.meta["DBMIN"] == min_samples_start
assert np.count_nonzero(saved_candidates["dbscan_core"]) == 1_154

print("Saved candidate catalogue passed all validation checks.")
print(f"Rows: {len(saved_candidates):,}")
print(f"Columns: {len(saved_candidates.colnames)}")

## 15. Results and limitations

Using a broad local astrometric window, standardized parallax and proper motion, and the provisional parameters `eps = 0.18` and `min_samples = 10`, DBSCAN identified 1,185 Pleiades candidates. Of these, 1,154 are core points and 31 are border points. The cluster centre and membership are stable across nearby parameter choices, and the candidates show an independent sky concentration near the Pleiades centre.

Important limitations remain:

- The broad analysis window was informed by the expected Pleiades location. Sources outside it cannot be recovered by this run.
- Standard scaling and DBSCAN parameters are modelling choices, not physical membership laws.
- DBSCAN does not use individual Gaia uncertainties or astrometric correlations.
- The elongated feature may include real perspective structure, binaries, measurement scatter, and field contamination.
- Core and border status describe local density, not membership probability.

Notebook 04 will provide the next scientific validation: genuine coeval Pleiades members should trace a coherent sequence in a Gaia colour–magnitude diagram. We should retain awareness that photometric binaries and missing BP/RP measurements can complicate that test.